# Stage 11 V2a — Otonom sembol koruma (CPU/GPU)

Tek notebook: **V2 best checkpoint → V2a fine-tune → development → gate geçerse held-out → Stage 9A**.

Bu sürüm varsayılan olarak **CPU** kullanır; Colab GPU kotası olmasa da çalışabilir. CPU eğitimi T4 GPU'ya göre belirgin biçimde daha yavaştır, fakat aynı checkpoint/provenance/gate düzeni korunur.

Eğitim arka plan süreci olarak başlatılır. Durum Google Drive'a yazılır; notebook/telefon ekranı yalnızca izleme için kullanılır. Colab runtime tamamen sonlandırılırsa süreç durur, ancak V2a ara checkpoint'lerinden yeniden devam eder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys, json, time
REPO = Path('/content/st-score-restore-engine')
REF = os.environ.get('ST_SCORE_RESTORE_REF', 'main')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',REF,'https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',REF,'--depth','1'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'], check=True)
import torch
os.environ['ST_SCORE_RESTORE_DEVICE'] = os.environ.get('ST_SCORE_RESTORE_DEVICE', 'cpu')
print('Python:', sys.version.split()[0])
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Seçilen eğitim cihazı:', os.environ['ST_SCORE_RESTORE_DEVICE'].upper())
if os.environ['ST_SCORE_RESTORE_DEVICE'].lower() == 'cpu':
    print('CPU mode: GPU kotası gerektirmez; eğitim T4 GPU’dan daha uzun sürebilir.')
V2 = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v2_symbol_preservation/best.pt')
if not V2.exists(): raise FileNotFoundError(V2)
print('V2 best.pt: OK')
print('Preflight: OK')


## 1. Otonom süreci arka planda başlat
Bu hücre birkaç saniyede geri döner. Aynı süreç zaten çalışıyorsa ikinci kez başlatmaz. CPU seçimi arka plan sürecine aktarılır.


In [ ]:
OUT = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v2a_symbol_preservation')
OUT.mkdir(parents=True, exist_ok=True)
PID = OUT/'pipeline.pid'
LOG = OUT/'pipeline.v2a.log'
def alive(pid):
    try:
        os.kill(pid, 0); return True
    except Exception:
        return False
existing = None
if PID.exists():
    try: existing = int(PID.read_text().strip())
    except: existing = None
if existing and alive(existing):
    print('V2a zaten çalışıyor. PID:', existing)
else:
    log = open(LOG, 'a', buffering=1)
    proc = subprocess.Popen([sys.executable,'-u',str(REPO/'tools'/'stage11_v2a_autonomous_pipeline.py')], stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=os.environ.copy())
    PID.write_text(str(proc.pid))
    print('V2a arka planda başlatıldı. PID:', proc.pid)
    print('Cihaz:', os.environ['ST_SCORE_RESTORE_DEVICE'].upper())
print('Log:', LOG)


## 2. İzleme ekranı
Bu hücre Drive'daki durum dosyalarını okur. İstediğiniz zaman durdurup tekrar çalıştırabilirsiniz; eğitim sürecini durdurmaz.


In [ ]:
STATUS = OUT/'run_status.v2a.json'
PIPE = OUT/'pipeline_status.v2a.json'
LOG = OUT/'pipeline.v2a.log'
for _ in range(360):
    print('\n' + '='*60)
    if PIPE.exists():
        try:
            p=json.loads(PIPE.read_text()); print('PIPELINE:', json.dumps(p, ensure_ascii=False, indent=2))
        except Exception as e: print('pipeline status okunamadı:', e)
    if STATUS.exists():
        try:
            s=json.loads(STATUS.read_text()); print('CANLI DURUM:', json.dumps(s, ensure_ascii=False, indent=2))
        except Exception as e: print('run status okunamadı:', e)
    if LOG.exists():
        lines=LOG.read_text(errors='replace').splitlines(); print('SON LOG:'); print('\n'.join(lines[-12:]))
    if PIPE.exists():
        try:
            stage=json.loads(PIPE.read_text()).get('stage','')
            if stage in {'complete','complete_review_required','failed'}:
                print('Pipeline sonlandı:', stage); break
        except: pass
    time.sleep(60)


## 3. Sonuç özeti
Pipeline bittikten sonra bu hücre sonucu tek ekranda gösterir.


In [ ]:
paths = {
 'pipeline': OUT/'pipeline_status.v2a.json',
 'training': OUT/'training_evidence.v2a.json',
 'dev': Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/deepscoresv2_dense_v2a_dev/development_evidence.v2a.json'),
 'heldout': Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/deepscoresv2_dense_v2a_heldout/heldout_evidence.v2a.json'),
 'stage9a': Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/deepscoresv2_dense_v2a_stage9a/stage9a_symbol_region_evidence.v2a.json'),
}
for name,path in paths.items():
    print('\n###', name, '###')
    if path.exists():
        try: print(json.dumps(json.loads(path.read_text()), indent=2, ensure_ascii=False))
        except: print(path.read_text()[-4000:])
    else: print('Henüz oluşmadı:', path)
